In [1]:
import os
import kagglehub
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt


# Download latest version
path = kagglehub.dataset_download("briscdataset/brisc2025")



In [2]:

data_path = os.path.join(path)

In [3]:
os.listdir(data_path)

['brisc2025']

In [4]:
print(type(os.listdir(data_path)))

<class 'list'>


In [ ]:

def analyze_class_type(classname: str) -> str:
  image_size = set()
  image_mode = set()
  image_format = set()
  overall_min_pixel = float('inf') 
  overall_max_pixel = float('-inf') 
  class_dir = os.path.join(data_path, "brisc2025","classification_task","train",classname)
  image_files = os.listdir(class_dir)
  file_size = len(image_files)
  for img_filename in image_files:
    img_path = os.path.join(class_dir, img_filename)
    images = Image.open(img_path)
    image_size.add(images.size)
    image_mode.add(images.mode)
    image_format.add(images.format)
    images_array = np.array(images)
    overall_min_pixel = min(overall_min_pixel, images_array.min())
    overall_max_pixel = max(overall_max_pixel, images_array.max())

  return {
    "total_images": file_size,
    "image_size": image_size,
    "image_mode": image_mode,
    "image_format": image_format,
    "min_pixel": overall_min_pixel,
    "max_pixel": overall_max_pixel
}

In [ ]:
for cls in os.listdir( os.path.join(data_path, "brisc2025","classification_task","train")):
  print(f"******** class Analyse Report for {cls}********")
  for key, values in analyze_class_type(cls).items():
    print(f"{key}: {values}")
  print("-"*100)

In [ ]:
train_data_path = os.path.join(data_path, "brisc2025","classification_task","train")
test_data_path = os.path.join(data_path, "brisc2025","classification_task","test")

In [ ]:
def get_all_image_paths(folder_path):
    image_paths = []

    for cls in os.listdir(folder_path):
        class_path = os.path.join(folder_path, cls)

        for file in os.listdir(class_path):
            image_paths.append(os.path.join(class_path, file))

    return image_paths

In [ ]:
def find_corrupted_images(folder_path:str) -> list:

    corrupted = []

    for file_path in get_all_image_paths(folder_path):

        try:
            img = Image.open(file_path)
            img.verify()

        except Exception:
            corrupted.append(file_path)

    return corrupted

In [ ]:
find_corrupted_images(train_data_path)

In [ ]:
from typing import Tuple

def find_empty_images(folder_path: str) -> Tuple[list, int]:
    empty_images = []

    for file_path in get_all_image_paths(folder_path):

        if os.path.getsize(file_path) == 0:
            empty_images.append(file_path)

    return empty_images, len(empty_images)

In [ ]:
empty_images, count = find_empty_images(train_data_path)

if count == 0:
    print("No empty image files found in the dataset.")
elif count >= 5:
    print(f"Found {count} empty image(s):")
    for img in empty_images:
        print(img)

In [ ]:
def analyze_intensity_distribution(folder_path:str, *args:str)-> dict:
  mean_value = {}
  class_path = os.path.join(folder_path,*args)
  for file in os.listdir(class_path):
    file_path = os.path.join(class_path, file)
    with Image.open(file_path) as image:
      image_array = np.array(image)
    mean_value[file] = image_array.mean()
  darkest_image = min(mean_value, key=mean_value.get)
  brightest_image = max(mean_value, key=mean_value.get)
  average = round(np.mean(list(mean_value.values())), 2)
  return {
    "Total Images": len(mean_value),
    "Average Mean Intensity": average ,
    "Darkest Image": darkest_image,
    "Darkest Image Mean": round(mean_value[darkest_image],2),
    "Brightest Image" : brightest_image,
    "Brightest Image Mean": round(mean_value[brightest_image],2)
}

In [ ]:
for cls in os.listdir(train_data_path):
  print(f"******** Intensity Report for {cls} ********")
  for key, values in analyze_intensity_distribution(train_data_path,cls).items():
    
    print(f"{key}: {values}")
  print("-"*100)
